In [1]:
from gurobipy import Model, GRB

In [2]:
# Create a model
m = Model("sensitivity_analysis")

# Create variables
x1 = m.addVar(lb=0, ub=0.50, name="stock")
x2 = m.addVar(lb=0.3, name="bond")
x3 = m.addVar(lb=0.1, name="real_est")

# Set objective
m.setObjective(0.08*x1 + 0.05*x2 +0.06*x3, GRB.MAXIMIZE)

# Add constraints
c1 = m.addConstr(x1 <= 0.5, "c1")
c2 = m.addConstr(x2 >= 0.30, "c2")
c3 = m.addConstr(x3 >= 0.1, "c3")
c4 = m.addConstr(x1 + x2 +x3 == 1, "c4")

# Optimize model
m.optimize()

# Check if the model has an optimal solution
if m.status == GRB.OPTIMAL:
    print("Optimal Solution:")
    print(f"x1: {x1.x}, x2: {x2.x}, x3: {x3.x}")

Set parameter Username
Academic license - for non-commercial use only - expires 2026-09-08
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 4 rows, 3 columns and 6 nonzeros
Model fingerprint: 0xb59f9a57
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e-02, 8e-02]
  Bounds range     [1e-01, 5e-01]
  RHS range        [1e-01, 1e+00]
Presolve removed 4 rows and 3 columns
Presolve time: 0.05s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.7000000e-02   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.07 seconds (0.00 work units)
Optimal objective  6.700000000e-02
Optimal Solution:
x1: 0.5, x2: 0.3, x3: 0.2


In [4]:
# Ranges in which the current basis remains optimal
print("\nRanges for RHS:") 
# Ranges for RHS within which the current basis stays optimal are found using c.SARHSLow and c.SARHSUp where c is a constraint.
for c in m.getConstrs():
    print(f"{c.ConstrName}: {c.SARHSLow} to {c.SARHSUp}")


Ranges for RHS:
c1: 0.5 to inf
c2: -inf to 0.3
c3: -inf to 0.2
c4: 0.9 to inf


In [5]:
# Sensitivity Analysis
print("\nSensitivity Analysis")

print("\nRanges for Objective Coefficients:")
for v in m.getVars():     # Ranges for Objective Coefficients within which the current basis stays optimal are found using v.SAObjLow and v.SAObjUp where v is a variable.
    print(f"{v.VarName}: {v.SAObjLow} to {v.SAObjUp}")



Sensitivity Analysis

Ranges for Objective Coefficients:
stock: 0.06 to inf
bond: -inf to 0.06
real_est: 0.05 to 0.08


In [6]:
#let's try a scenario where we increase the RHS of constraint 1 (c1) by 0.1 unit

# Modify the RHS of a constraint and reoptimize
rhs_change = 0.1  # Change this value as needed
c1.RHS = 0.5 - rhs_change  # Changing the RHS of constraint c1

m.update()

# Reoptimize the model with the modified constraint
m.optimize()

# Check if the model has an optimal solution
if m.status == GRB.OPTIMAL:
    print("Optimal Solution:")
    print(f"x1: {x1.x}, x2: {x2.x}, x3: {x3.x}")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 4 rows, 3 columns and 6 nonzeros
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e-02, 8e-02]
  Bounds range     [1e-01, 5e-01]
  RHS range        [1e-01, 1e+00]
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.7000000e-02   1.000000e-01   0.000000e+00      0s
       1    6.5000000e-02   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.02 seconds (0.00 work units)
Optimal objective  6.500000000e-02
Optimal Solution:
x1: 0.4, x2: 0.3, x3: 0.29999999999999993


In [6]:
# Adding a new constraint and reoptimizing
c5 = m.addConstr(x1 + 2*x2 <= 0.9, "c5")  # To add a new constraint and reoptimize, simply use the addConstr() method followed by optimize().
m.update()
m.optimize()
print("\nSolution after adding a new constraint:")

print(f"x1: {x1.x}, x2: {x2.x}, x3: {x3.x}")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 5 rows, 3 columns and 8 nonzeros
Coefficient statistics:
  Matrix range     [1e+00, 2e+00]
  Objective range  [5e-02, 8e-02]
  Bounds range     [1e-01, 5e-01]
  RHS range        [1e-01, 1e+00]
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.5000000e-02   5.000000e-02   0.000000e+00      0s
       1    6.3000000e-02   0.000000e+00   0.000000e+00      0s

Solved in 1 iterations and 0.01 seconds (0.00 work units)
Optimal objective  6.300000000e-02

Solution after adding a new constraint:
x1: 0.30000000000000004, x2: 0.3, x3: 0.3999999999999999


In [7]:
#Remove a constraint and reoptimize
m.remove(c5)
m.update()
m.optimize()
print("\nSolution after adding a new variable:")
print(f"x1: {x1.x}, x2: {x2.x}, x3: {x3.x}")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 4 rows, 3 columns and 6 nonzeros
Model fingerprint: 0x2eb66b56
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [5e-02, 8e-02]
  Bounds range     [1e-01, 5e-01]
  RHS range        [1e-01, 1e+00]
Presolve removed 4 rows and 3 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
       0    6.5000000e-02   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  6.500000000e-02

Solution after adding a new variable:
x1: 0.4, x2: 0.3, x3: 0.29999999999999993


In [8]:
# Adding a new variable and reoptimizing
x4 = m.addVar(lb=0, name="x4")
c6 = m.addConstr(x1 + x2 + x3 + x4 <= 1.2, "c6")
m.setObjective(3*x1 + 2*x2 + x3 +x4, GRB.MAXIMIZE)
m.update()

m.optimize()
print("\nSolution after adding a new variable:")
print(f"x1: {x1.x}, x2: {x2.x}, x3: {x3.x}, , x4: {x4.x}")

Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (win64)

CPU model: 13th Gen Intel(R) Core(TM) i9-13900H, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 5 rows, 4 columns and 10 nonzeros
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 3e+00]
  Bounds range     [1e-01, 5e-01]
  RHS range        [1e-01, 1e+00]
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+30   3.000000e+30   2.000000e+00      0s
       2    2.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 2 iterations and 0.01 seconds (0.00 work units)
Optimal objective  2.500000000e+00

Solution after adding a new variable:
x1: 0.4, x2: 0.5, x3: 0.1, , x4: 0.19999999999999984
